# Fine-tune PhoBERT on Kaggle GPU

Before running:
1. **Settings → Accelerator → GPU T4 x2** (or P100), and **Internet → On**.
2. **Add Input** → your private dataset with `train` / `val` / `test` CSVs (columns `text,label`).
3. **Add-ons → Secrets** → add `HF_TOKEN` (a *write* token from huggingface.co/settings/tokens) and attach it to this notebook.

Then **Run All**. Two experiments are trained (without / with teencode normalization, ~30–45 min each),
and the better one (by validation F1-macro) is pushed to the Hugging Face Hub.

In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git
%cd vietnamese-toxic-comment-moderation
!pip install -q pyvi emoji
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
# Copy the CSVs from the Kaggle input dataset into data/raw/{train,val,test}.csv
import glob, shutil, os
os.makedirs("data/raw", exist_ok=True)
for split in ["train", "val", "test"]:
    matches = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if split in os.path.basename(p).lower()]
    assert len(matches) == 1, f"Expected exactly one CSV for '{split}', found: {matches}"
    shutil.copy(matches[0], f"data/raw/{split}.csv")
    print(split, "<-", matches[0])

In [ ]:
# Hugging Face login using the Kaggle secret
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
# Experiment A: no teencode normalization
!python scripts/train_phobert.py --output-dir models/phobert_A --no-push

In [ ]:
# Experiment B: with teencode normalization
!python scripts/train_phobert.py --teencode --output-dir models/phobert_B --no-push

In [ ]:
# Compare and push the better model (by validation F1-macro, never by test)
import json
a = json.load(open("reports/phobert_results.json"))
b = json.load(open("reports/phobert_results_teencode.json"))
for name, r in [("A (no teencode)", a), ("B (teencode)", b)]:
    print(f"{name}: val F1={r['val_f1_macro']} | test F1={r['test']['f1_macro']} acc={r['test']['accuracy']} per-class={r['test']['f1_per_class']}")

best_dir = "models/phobert_B/best" if b["val_f1_macro"] > a["val_f1_macro"] else "models/phobert_A/best"
print("Pushing", best_dir)
from huggingface_hub import HfApi
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
api = HfApi()
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path=best_dir, repo_id=repo_id)
print(f"https://huggingface.co/{repo_id}")

## Send the results back

Run the next cell, **copy the whole output** and paste it into the chat with Claude
(it is used to update the README results table).

In [ ]:
print(json.dumps({"A_no_teencode": a, "B_teencode": b}, ensure_ascii=False))